<div align="center">
  <a href="https://www.youtube.com/@aigolden?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
</div>

In [ ]:
#@title ⚙️ Install
!pip install -q --upgrade gemini-srt-translator yt-dlp
print("✅ Done!")

In [ ]:
#@title 🔑 Configuration
Gemini_API = "" #@param {type:"string"}

import os
os.environ['GEMINI_API_KEY'] = Gemini_API
print("✅ Done!")

In [ ]:

#@title 📂 Input
Input_Method = "آپلود از دستگاه" #@param ["آپلود از دستگاه", "لینک یوتیوب", "گوگل درایو"]
YT_Link = "" #@param {type:"string"}

import subprocess, os
from google.colab import files

AUDIO_FILE = None
OUTPUT_FILE = None
SUBTITLE_READY = False

def compress_to_mp3(full_path):
    fname = os.path.basename(full_path)
    mp3_name = fname.rsplit('.', 1)[0] + '_compressed.mp3'
    subprocess.run(
        ["ffmpeg", "-i", full_path, "-acodec", "libmp3lame", "-ab", "64k", "-ac", "1", mp3_name, "-y"],
        capture_output=True
    )
    return mp3_name

# ---------- حالت ۱: آپلود از دستگاه ----------
if Input_Method == "آپلود از دستگاه":
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    print(f"🔄 در حال فشرده‌سازی {fname}...")
    AUDIO_FILE = compress_to_mp3(fname)
    print(f"✅ فایل فشرده آماده شد: {AUDIO_FILE}")

# ---------- حالت ۲: لینک یوتیوب ----------
elif Input_Method == "لینک یوتیوب":
    if not YT_Link.strip():
        print("❌ لینک یوتیوب وارد نشده!")
    else:
        print("🔍 بررسی زیرنویس ویدیو...")
        check = subprocess.run(["yt-dlp", "--list-subs", YT_Link], capture_output=True, text=True)
        has_sub = "has no subtitles" not in check.stdout and "Language" in check.stdout

        if has_sub:
            print("✅ زیرنویس پیدا شد! در حال دانلود...")
            subprocess.run(
                ["yt-dlp", "--write-subs", "--write-auto-subs", "--sub-format", "srt",
                 "--convert-subs", "srt", "--skip-download", "--output", "yt_audio.%(ext)s", YT_Link],
                capture_output=True
            )
            srt_files = [f for f in os.listdir('.') if f.startswith('yt_audio') and f.endswith('.srt')]
            if srt_files:
                os.rename(srt_files[0], 'yt_audio.srt')
                AUDIO_FILE = "yt_audio.mp3"
                SUBTITLE_READY = True
                print("✅ زیرنویس دانلود شد: yt_audio.srt")
                print("⚠️ نیازی به اجرای سلول Transcribe نیست — سلول Download را اجرا کنید.")
            else:
                print("⚠️ زیرنویس پیدا نشد، در حال دانلود صدا برای Gemini...")

        if not SUBTITLE_READY:
            print("🎵 در حال دانلود صدا با کیفیت و حجم بهینه...")
            result = subprocess.run(
                ["yt-dlp", "--extract-audio", "--audio-format", "mp3", "--audio-quality", "5",
                 "--postprocessor-args", "ExtractAudio:-ac 1",
                 "--output", "yt_audio.%(ext)s", "--print", "after_move:filepath", YT_Link],
                capture_output=True, text=True
            )
            if result.returncode != 0:
                print(f"❌ خطا: {result.stderr}")
            else:
                AUDIO_FILE = "yt_audio.mp3"
                size_mb = os.path.getsize(AUDIO_FILE) / 1024 / 1024
                print(f"✅ دانلود شد: {AUDIO_FILE} ({size_mb:.1f} MB)")
                print("▶️ حالا سلول Transcribe را اجرا کنید.")

# ---------- حالت ۳: گوگل درایو (فایل‌منیجر تعاملی) ----------
elif Input_Method == "گوگل درایو":
    from google.colab import drive
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    if not os.path.ismount('/content/drive'):
        print("🔗 در حال اتصال به گوگل درایو...")
        drive.mount('/content/drive')

    AUDIO_EXTS = ('.mp3', '.wav', '.m4a', '.aac', '.flac', '.ogg', '.mp4', '.mkv', '.mov')
    current_dir = ['/content/drive/MyDrive']

    out = widgets.Output()
    nav_dropdown = widgets.Dropdown(description='موارد:', layout=widgets.Layout(width='500px'))
    open_btn = widgets.Button(description='باز کردن 📂', button_style='info')
    select_btn = widgets.Button(description='انتخاب فایل ✅', button_style='success')
    up_btn = widgets.Button(description='⬆️ بازگشت')
    path_label = widgets.Label()

    def refresh_list():
        try:
            entries = os.listdir(current_dir[0])
        except Exception:
            entries = []
        entries = sorted(entries, key=lambda e: (not os.path.isdir(os.path.join(current_dir[0], e)), e.lower()))
        options = []
        for e in entries:
            full = os.path.join(current_dir[0], e)
            if os.path.isdir(full):
                options.append(f"📁 {e}")
            elif e.lower().endswith(AUDIO_EXTS):
                options.append(f"🎵 {e}")
        nav_dropdown.options = options if options else ["(پوشه خالی است)"]
        path_label.value = f"📍 مسیر فعلی: {current_dir[0].replace('/content/drive/MyDrive', 'My Drive')}"

    def on_open(b):
        if not nav_dropdown.value or not nav_dropdown.value.startswith("📁"):
            return
        full = os.path.join(current_dir[0], nav_dropdown.value[2:])
        current_dir[0] = full
        refresh_list()

    def on_up(b):
        parent = os.path.dirname(current_dir[0].rstrip('/'))
        if parent.startswith('/content/drive'):
            current_dir[0] = parent
            refresh_list()

    def on_select(b):
        global AUDIO_FILE
        with out:
            clear_output()
            if not nav_dropdown.value or not nav_dropdown.value.startswith("🎵"):
                print("⚠️ لطفاً یک فایل صوتی/ویدیویی انتخاب کنید (نه پوشه).")
                return
            fname = nav_dropdown.value[2:]
            full = os.path.join(current_dir[0], fname)
            print(f"🔄 در حال آماده‌سازی: {fname}")
            AUDIO_FILE = compress_to_mp3(full)
            print(f"✅ فایل انتخاب شد: {AUDIO_FILE}")
            print("▶️ حالا سلول Transcribe را اجرا کنید.")

    open_btn.on_click(on_open)
    up_btn.on_click(on_up)
    select_btn.on_click(on_select)

    refresh_list()
    display(widgets.VBox([path_label, nav_dropdown, widgets.HBox([up_btn, open_btn, select_btn]), out]))

In [ ]:

#@title 🎙️ Transcribe
Output_Format = "srt" #@param ["srt", "txt"]

import gemini_srt_translator as gst
import os, subprocess, math, time, sys, tempfile, re
from datetime import timedelta, datetime

MODEL_CHAIN = [
    {"name": "gemini-3.6-flash",         "rpm": 5,  "rpd": 20},
    {"name": "gemini-3.5-flash-lite",    "rpm": 15, "rpd": 500},
    {"name": "gemini-3.5-flash",         "rpm": 5,  "rpd": 20},
    {"name": "gemini-3.1-flash-lite",    "rpm": 15, "rpd": 500},
    {"name": "gemini-flash-latest",      "rpm": 10, "rpd": 20},
    {"name": "gemini-flash-lite-latest", "rpm": 10, "rpd": 20},
]
model_state = {m["name"]: {"times": [], "daily": 0, "day": datetime.now().date()} for m in MODEL_CHAIN}
current_idx = 0

def model_name():
    return MODEL_CHAIN[current_idx]["name"]

def can_use(idx):
    m = MODEL_CHAIN[idx]
    st = model_state[m["name"]]
    today = datetime.now().date()
    if st["day"] != today:
        st["day"], st["daily"], st["times"] = today, 0, []
    if st["daily"] >= m["rpd"]:
        return False
    recent = [t for t in st["times"] if time.time() - t < 60]
    return len(recent) < m["rpm"]

def record(idx):
    st = model_state[MODEL_CHAIN[idx]["name"]]
    st["times"].append(time.time())
    st["daily"] += 1

def advance():
    global current_idx
    if current_idx < len(MODEL_CHAIN) - 1:
        current_idx += 1
        print(f"🔄 سوییچ به مدل {model_name()}")
        return True
    return False

QUOTA_MARKERS = ["quota", "exceeded", "429", "resource_exhausted", "rate limit"]

def run_transcribe(api_key, model, audio_path, output_path):
    script = (
        "import gemini_srt_translator as gst\n"
        f"gst.gemini_api_key = {api_key!r}\n"
        f"gst.model_name = {model!r}\n"
        f"gst.audio_file = {audio_path!r}\n"
        f"gst.output_file = {output_path!r}\n"
        "gst.transcribe()\n"
    )
    script_path = tempfile.mktemp(suffix=".py")
    with open(script_path, "w") as f:
        f.write(script)

    proc = subprocess.Popen(
        [sys.executable, script_path],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    quota_hit = False
    try:
        for line in proc.stdout:
            print(line, end="")
            if any(marker in line.lower() for marker in QUOTA_MARKERS):
                quota_hit = True
                proc.kill()
                break
    finally:
        proc.wait()
        try:
            os.remove(script_path)
        except OSError:
            pass

    if quota_hit:
        return "quota"
    return "success" if proc.returncode == 0 else "error"

def parse_time(t):
    h, m, s_ms = t.split(':')
    s, ms = s_ms.split(',')
    return timedelta(hours=int(h), minutes=int(m), seconds=int(s), milliseconds=int(ms))

def format_time(td):
    total_ms = int(td.total_seconds() * 1000)
    h, rem = divmod(total_ms, 3600000)
    m, rem = divmod(rem, 60000)
    s, ms = divmod(rem, 1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"

def get_duration(path):
    out = subprocess.run(
        ["ffprobe", "-i", path, "-show_entries", "format=duration", "-v", "quiet", "-of", "csv=p=0"],
        capture_output=True, text=True
    ).stdout.strip()
    return float(out) if out else 0.0

def detect_silences(path, noise_db="-30dB", min_dur=0.5):
    """بازه‌های سکوت را در فایل صوتی برمی‌گرداند: [(start, end), ...]"""
    result = subprocess.run(
        ["ffmpeg", "-i", path, "-af", f"silencedetect=noise={noise_db}:d={min_dur}",
         "-f", "null", "-"],
        capture_output=True, text=True
    )
    log = result.stderr
    starts = [float(x) for x in re.findall(r"silence_start:\s*([\d.]+)", log)]
    ends = [float(x) for x in re.findall(r"silence_end:\s*([\d.]+)", log)]
    return list(zip(starts, ends))

def find_cut_points(total_duration, num_chunks, silences, search_window=30):
    """نقاط برش نهایی را بر اساس نزدیک‌ترین سکوت به هر نقطه‌ی هدف پیدا می‌کند."""
    if num_chunks <= 1:
        return [0.0, total_duration]

    raw_targets = [total_duration * i / num_chunks for i in range(1, num_chunks)]
    cut_points = [0.0]
    for target in raw_targets:
        best = None
        best_dist = search_window
        for s, e in silences:
            mid = (s + e) / 2
            dist = abs(mid - target)
            if dist <= best_dist:
                best_dist = dist
                best = mid
        cut_points.append(best if best is not None else target)
    cut_points.append(total_duration)
    return sorted(set(cut_points))

if AUDIO_FILE is None:
    print("❌ ابتدا سلول Input را اجرا کنید!")
else:
    base_name = AUDIO_FILE.rsplit('.', 1)[0]
    OUTPUT_FILE = f"{base_name}.{Output_Format}"
    chunk_dir = f"{base_name}_chunks"
    os.makedirs(chunk_dir, exist_ok=True)

    total_duration = get_duration(AUDIO_FILE)
    total_minutes = total_duration / 60
    num_chunks = 1 if total_minutes <= 20 else math.ceil(total_minutes / 20)
    print(f"⏱️ مدت فایل: {total_minutes:.1f} دقیقه → تقسیم به {num_chunks} تکه")

    existing_chunks = sorted([f for f in os.listdir(chunk_dir) if f.endswith('.mp3')])
    if not existing_chunks:
        if num_chunks == 1:
            subprocess.run(["ffmpeg", "-i", AUDIO_FILE, "-c", "copy",
                             os.path.join(chunk_dir, "chunk_000.mp3"), "-y"], capture_output=True)
            cut_points = [0.0, total_duration]
        else:
            print("🔍 در حال شناسایی سکوت‌های بین دیالوگ‌ها...")
            silences = detect_silences(AUDIO_FILE)
            cut_points = find_cut_points(total_duration, num_chunks, silences)
            print(f"✂️ در حال برش فایل روی {len(cut_points)-1} نقطه‌ی سکوت...")
            for i in range(len(cut_points) - 1):
                start, end = cut_points[i], cut_points[i + 1]
                out_path = os.path.join(chunk_dir, f"chunk_{i:03d}.mp3")
                subprocess.run(
                    ["ffmpeg", "-i", AUDIO_FILE, "-ss", str(start), "-to", str(end),
                     "-c", "copy", out_path, "-y"],
                    capture_output=True
                )
        existing_chunks = sorted([f for f in os.listdir(chunk_dir) if f.endswith('.mp3')])
        print(f"✅ {len(existing_chunks)} تکه ساخته شد (برش‌ها روی نقاط سکوت انجام شد).")

    durations = [get_duration(os.path.join(chunk_dir, f)) for f in existing_chunks]
    offsets = [sum(durations[:i]) for i in range(len(durations))]

    api_key = os.environ['GEMINI_API_KEY']
    success = True

    for i, chunk_file in enumerate(existing_chunks):
        chunk_path = os.path.join(chunk_dir, chunk_file)
        chunk_out = chunk_path.rsplit('.', 1)[0] + f".{Output_Format}"

        if os.path.exists(chunk_out) and os.path.getsize(chunk_out) > 0:
            print(f"✅ تکه {i+1}/{len(existing_chunks)} قبلاً آماده بود، رد شد.")
            continue

        done = False
        while not done:
            while not can_use(current_idx):
                print(f"⏳ مدل {model_name()} به سقف مجاز رسیده...")
                if not advance():
                    print("❌ همه‌ی مدل‌ها به سقف رسیدن. کمی صبر کنید یا فردا دوباره اجرا کنید.")
                    success = False
                    break
            if not success:
                break

            print(f"🎙️ تکه {i+1}/{len(existing_chunks)} با مدل {model_name()} ...")
            if os.path.exists(chunk_out):
                os.remove(chunk_out)

            result = run_transcribe(api_key, model_name(), chunk_path, chunk_out)

            if result == "success" and os.path.exists(chunk_out) and os.path.getsize(chunk_out) > 0:
                record(current_idx)
                print(f"✅ تکه {i+1} آماده شد.")
                done = True
            elif result == "quota":
                print(f"⚠️ مدل {model_name()} به quota خورد، سوییچ فوری...")
                if not advance():
                    print("❌ همه‌ی مدل‌ها quota‌شون تموم شده.")
                    success = False
                    break
            else:
                print(f"⚠️ خطای نامشخص با مدل {model_name()}.")
                if not advance():
                    success = False
                    break

        if not success:
            print("⚠️ فقط همین سلول رو دوباره اجرا کنید — تکه‌های آماده رد می‌شن و کار ادامه پیدا می‌کنه.")
            break

    if success:
        print("🔗 در حال ادغام تکه‌ها...")
        if Output_Format == "srt":
            all_blocks = []
            for i, chunk_file in enumerate(existing_chunks):
                chunk_out = os.path.join(chunk_dir, chunk_file).rsplit('.', 1)[0] + ".srt"
                with open(chunk_out, encoding='utf-8') as f:
                    blocks = f.read().strip().split('\n\n')
                for b in blocks:
                    lines = b.split('\n')
                    if len(lines) < 2:
                        continue
                    start, end = lines[1].split(' --> ')
                    lines[1] = (f"{format_time(parse_time(start) + timedelta(seconds=offsets[i]))} --> "
                                f"{format_time(parse_time(end) + timedelta(seconds=offsets[i]))}")
                    all_blocks.append('\n'.join(lines))
            final = []
            for idx, b in enumerate(all_blocks, 1):
                lines = b.split('\n')
                lines[0] = str(idx)
                final.append('\n'.join(lines))
            with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
                f.write('\n\n'.join(final) + '\n')
        else:
            with open(OUTPUT_FILE, 'w', encoding='utf-8') as out_f:
                for chunk_file in existing_chunks:
                    chunk_out = os.path.join(chunk_dir, chunk_file).rsplit('.', 1)[0] + ".txt"
                    with open(chunk_out, encoding='utf-8') as f:
                        out_f.write(f.read().strip() + "\n\n")
        print(f"✅ ذخیره شد: {OUTPUT_FILE}")

In [ ]:

#@title 💾 Download
from google.colab import files
from IPython.display import display, HTML
import os

def show_subscribe_banner():
    image_url = 'https://huggingface.co/Toolsai/dubtest/resolve/main/newgolden.png'
    youtube_channel_url = 'https://youtube.com/@aigolden'
    html_code = f'''
    <div style="text-align: center; border: 2px solid #e0e0e0; padding: 15px; border-radius: 12px; background-color: #f9f9f9; max-width: 350px; margin: auto;">
        <a href="{youtube_channel_url}" target="_blank" title="رفتن به کانال یوتیوب AIGOLDEN">
            <img src="{image_url}" alt="AIGOLDEN YouTube Channel" style="max-width: 100%; height: auto; border-radius: 8px;">
        </a>
        <p style="font-size: 16px; font-family: 'Vazir', sans-serif; margin-top: 15px; color: #333;">
            برای مشاهده آموزش‌های بیشتر، ما را در یوتیوب دنبال کنید.
        </p>
        <a href="{youtube_channel_url}" target="_blank" style="text-decoration: none; display: inline-block; background-color: #FF0000; color: white; padding: 10px 20px; border-radius: 8px; font-weight: bold; font-family: 'Vazir', sans-serif; margin-top: 10px;">
            🚀 دنبال کردن در یوتیوب
        </a>
    </div>
    '''
    display(HTML(html_code))

if SUBTITLE_READY:
    files.download("yt_audio.srt")
    show_subscribe_banner()
elif OUTPUT_FILE and os.path.exists(OUTPUT_FILE):
    print(f"📥 دانلود: {OUTPUT_FILE}")
    files.download(OUTPUT_FILE)
    show_subscribe_banner()
else:
    print("❌ فایل خروجی پیدا نشد.")